# 04. Output Parser 

LLM 기본 출력은 문자열. 객체 생성 후 parser 지정하면 return된 내용이 해당 format에 맞게 출력.

LLM의 출력을 원하는 형식(문자열, JSON, 객체 등)으로 파싱합니다.

# LLM 객체 생성

In [1]:
import sys
sys.path.append("..")

from common_config import llm_connect
model = "gpt-5.4"

temperatuyre = 0
max_tokens = 2086

llm = llm_connect(model = model)

## 1. StrOutputParser - 문자열 추출

`AIMessage` 객체에서 `.content` 문자열만 추출합니다.

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}을 한 문장으로 설명해줘")
])

parser = StrOutputParser()

# prompt -> llm -> parser 순으로 수동 호출
messages = prompt.invoke({"topic": "LangChain"}) # prompt에 키워드 넣어서 나온 문장 message에 넣음.
response = llm.invoke(messages) # message로 llm과 통신해 respose에 넣음.
result = parser.invoke(response) # parser 객체에 respose를 넣어서 결과 얻음. llm에 안 갖다 옴. response의 conent 부분만 추출함.

print(type(result))  # str
print(result)

<class 'langchain_core.messages.base.TextAccessor'>
LangChain은 대형 언어 모델(LLM)을 외부 데이터, 도구, 워크플로와 연결해 실제 애플리케이션을 쉽게 만들 수 있게 해주는 개발 프레임워크입니다.


## 2. JsonOutputParser - JSON 파싱

LLM이 JSON을 반환하도록 유도하고 딕셔너리로 파싱합니다.

In [7]:
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("human", "파이썬 언어에 대한 정보를 name, year, creator 키를 가진 JSON으로 반환해줘. JSON만 반환해.")
])

parser = JsonOutputParser()

messages = prompt.invoke({})    # 넘겨주는 값 없으면 빈 딕셔너리 넘겨야 함. 이거 없으면 오류남.
response = llm.invoke(messages) # llm에 message 넣으면 response 나옴.
result = parser.invoke(response) # JsonOutputparser로 출력됨.

print(type(result))  # dict
print(result)

<class 'dict'>
{'name': 'Python', 'year': 1991, 'creator': 'Guido van Rossum'}


## 3. PydanticOutputParser - 구조체 파싱

Pydantic 모델을 정의하면 자동으로 포맷 지시사항을 생성하고 파싱합니다.

In [9]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Pydantic : 구조체 정의에 사용. Validation check 가능.

# 원하는 데이터의 출력 구조 정의
class MovieReview(BaseModel):
    title: str = Field(description="영화 제목")
    score: int = Field(description="10점 만점 점수")
    summary: str = Field(description="한 줄 감상")

parser = PydanticOutputParser(pydantic_object=MovieReview) # 사용자가 정의한 구조체를 정의하여 사용자 정의 parser 형태로 나옴.

# 파서가 자동으로 포맷 지시사항 생성
print(parser.get_format_instructions())

# 위에 정의된 설명 부분이 들어간 뒤 실질적 대답은 output schema 파트에 들어가 있음.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "영화 제목", "title": "Title", "type": "string"}, "score": {"description": "10점 만점 점수", "title": "Score", "type": "integer"}, "summary": {"description": "한 줄 감상", "title": "Summary", "type": "string"}}, "required": ["title", "score", "summary"]}
```


In [10]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 영화 평론가입니다.\n{format_instructions}"), # {format_instructions} : 이런 구조체로 영화평 작성하라.
    ("human",  "{movie} 영화를 리뷰해줘")
])

messages = prompt.invoke({
    "movie": "인터스텔라",
    "format_instructions": parser.get_format_instructions()
})
response = llm.invoke(messages) # llm에 messages 내용을 넣어 llm model이 생성한 response 얻음.
result   = parser.invoke(response)

print(type(result))    # <class MovieReview>
print(result.title)    # 인터스텔라
print(result.score)    # 10
print(result.summary)
print(result)

<class '__main__.MovieReview'>
인터스텔라
9
장대한 우주 과학과 깊은 가족애를 결합해, 다소 복잡한 설정에도 압도적인 감정과 스케일로 오래 남는 SF 걸작이다.
title='인터스텔라' score=9 summary='장대한 우주 과학과 깊은 가족애를 결합해, 다소 복잡한 설정에도 압도적인 감정과 스케일로 오래 남는 SF 걸작이다.'


## 4. CommaSeparatedListOutputParser - 리스트 파싱

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("human", "파이썬 웹 개발 프레임워크 Top 5를 쉼표로 구분해서 나열해줘. 설명 없이 이름만.")
])

# Prompt -> LLM -> Response 순서로 항상 진행되는 파이프라인. 이걸 파트 5에서 간략하게 표현하는 방법을 소개함.
messages = prompt.invoke({})     # 넘겨주는 값 없을 때 빈 list로
response = llm.invoke(messages)  # llm model에 message 넣고 그 결과를 response에 넣음.
result = parser.invoke(response) # 리스트 형태로 parsing해서 result 얻음.

print(type(result))  # list
print(result)

<class 'list'>
['Django', 'Flask', 'FastAPI', 'Pyramid', 'Tornado']
